<a href="https://colab.research.google.com/github/tharold4096/Tobias_INFO4670_Fall2026/blob/main/Copy_of_Week5_INFO4670_Assignment2_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [23]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [24]:
# TODO: set n_missing_study to the number of blank study_hours_reported values

missing = students.isna().sum()
print(missing[missing > 0])
n_missing = students["study_hours_reported"].isna().sum()
print(f"\nstudy_hours_reported: {n_missing} blank of {len(students)} "
      f"({n_missing/len(students)*100:.1f}%) -> {students['study_hours_reported'].notna().sum()} present")
n_missing_study = n_missing

study_hours_reported    255
dtype: int64

study_hours_reported: 255 blank of 2027 (12.6%) -> 1772 present


In [25]:
#Scratch cell

print(students.columns.tolist())
print(students.groupby("major")["study_hours_reported"].agg(["count", "median"]))
print(students.groupby("housing")["study_hours_reported"].agg(["count", "median"]))

['student_id', 'major', 'age', 'housing', 'commute_miles', 'work_hours_per_week', 'advisor_meetings', 'credits_attempted', 'final_gpa', 'study_hours_reported', 'enrollment_date', 'dropped_out']
                        count  median
major                                
Business Analytics        220   10.10
Cyber Security             69   10.90
Cybersecurity             463   10.70
Data Science              250   10.45
DataScience                34   10.35
Info Technology            29   10.00
Information  Science       29    9.40
Information Science       242   10.60
Information Technology    248   10.60
Learning Technologies     188   10.45
             count  median
housing                   
 On-Campus      32   11.30
Off-Campus     657   10.50
Off-campus      68   10.45
On-Campus      412   10.70
With Family    377   10.20
off campus     103   10.70
on-campus       60   11.15
with family     63   10.00


**Your justification (1–2 sentences):** _..._

I would choose to use the global median, simply because the catagories that are most likely to impact study_hours reported are not varying by a significant amount between groups. They are all realtively the same spread, so a global median would not cause a huge difference in calculation at the end.



In [26]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [27]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
# print(students.groupby("housing")["study_hours_reported"].agg(["count", "median"]))
students_clean = students.copy()
students_clean["housing_clean"] = students_clean["housing"].str.strip()
students_clean["housing_clean"] = students_clean["housing_clean"].str.lower()
students_clean["housing_clean"] = students_clean["housing_clean"].str.replace("-", " ")
print(students_clean["housing_clean"])
students["housing_clean"] = students_clean["housing_clean"]
print(students_clean["housing"].nunique())

0        off campus
1       with family
2       with family
3         on campus
4        off campus
           ...     
2022     off campus
2023    with family
2024    with family
2025     off campus
2026      on campus
Name: housing_clean, Length: 2027, dtype: object
8


In [28]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'off campus': 945, 'on campus': 590, 'with family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [30]:
# TODO
neg_work_mask = students["work_hours_per_week"] < 0
n_neg_work = int(neg_work_mask.sum())

age_mask = (students["age"] < 15) | (students["age"] > 90)
impossible_ages = sorted(students.loc[age_mask, "age"].unique())

In [31]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [35]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset="student_id", keep="first")

print("rows:", len(students_dedup), "| unique ids:", students_dedup["student_id"].nunique())

rows: 2000 | unique ids: 2000


**Why not de-dupe enroll / activity? (1 sentence):** _..._

Because both of those columns by nature will have multiple of the same value- multiple students can be enrolled in the same course, or participate in the same activity, and removing those duplicates would complete delete essentail data.

In [36]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [48]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
analysis = None
students_dedup = students_dedup.copy()
students_dedup["student_key"] = (students_dedup["student_id"].str.replace("NU-", "", regex=False).astype(int))
activity_summary = (
    activity.groupby("student_id")
    .agg(
        total_minutes=("minutes_active", "sum"),
        weeks_active=("minutes_active", "count"),
    )
    .reset_index()
)
analysis = students_dedup.merge(
    activity_summary,
    on="student_id",
    how="left",
    indicator=True,
)


In [49]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [53]:
# TODO
set_of_keys = set(analysis["student_key"])
matched = int(enroll["sid"].isin(set_of_keys).sum())

In [54]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [61]:
# TODO
before = analysis["enrollment_date"].isna().sum()
dates_parsed = pd.to_datetime( analysis["enrollment_date"], format="mixed", errors="coerce")
after = dates_parsed.isna().sum()
analysis["enrollment_date"] = dates_parsed


In [56]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [57]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
study = analysis["study_hours_reported"]
analysis["study_z"] = (study - study.mean()) / study.std()
analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[-0.01, 1, 2, 3, 4],
    labels=["0-1", "1-2", "2-3", "3-4"],
)


In [58]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [62]:
# TODO: write analysis to northgate_clean.csv
print(analysis.columns.tolist())
print(analysis.dtypes)
analysis = analysis.drop(columns="_merge")
analysis.to_csv("northgate_clean.csv", index=False)



['student_id', 'major', 'age', 'housing', 'commute_miles', 'work_hours_per_week', 'advisor_meetings', 'credits_attempted', 'final_gpa', 'study_hours_reported', 'enrollment_date', 'dropped_out', 'housing_clean', 'student_key', 'total_minutes', 'weeks_active', '_merge', 'study_z', 'gpa_band']
student_id                      object
major                           object
age                              int64
housing                         object
commute_miles                  float64
work_hours_per_week              int64
advisor_meetings                 int64
credits_attempted                int64
final_gpa                      float64
study_hours_reported           float64
enrollment_date         datetime64[ns]
dropped_out                      int64
housing_clean                   object
student_key                      int64
total_minutes                    int64
weeks_active                     int64
_merge                        category
study_z                        float64
gpa_ba

In [63]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- Missing values: study hours had 255 missing values, I used global median instead of any other of Han's method so that the calculations for average would not be skewed. There were no major fluctuations, so this was deemed acceptable.
- Housing: The eight catagories were cleaned into three using strip, lower, and replace of \- so that all the columns condensed down to one
- Impossible values: negative hours per week were tossed, while possible extremes like commute_miles were kept
- Duplicates: duplicates of student_id (the 18 exact + 9 extra) were tossed, student_id was chosen because it should be unique per student. Enrollment and activities were not tossed on duplicates because multiple students can be enrolled in the same of those
- Key: differetn key formats were used, so the NU prefix was dropped in order to merge them all together properly
- Dates: the date column contained many formats, so each value was parsed on its own, and analyzed to fit with the rest of the dates.


### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [68]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
print("mean GPA (clean):", analysis["final_gpa"].mean().round(3))
print("mean GPA (raw)  :", students["final_gpa"].mean().round(3))
print(students.groupby("housing")["final_gpa"].agg(["count", "mean"]))
print(analysis.groupby("housing_clean")["final_gpa"].agg(["count", "mean"]))

mean GPA (clean): 2.305
mean GPA (raw)  : 2.307
             count      mean
housing                     
 On-Campus      41  2.157073
Off-Campus     755  2.333391
Off-campus      77  2.361169
On-Campus      480  2.270250
With Family    420  2.305190
off campus     113  2.309027
on-campus       69  2.399855
with family     72  2.234444
               count      mean
housing_clean                 
off campus       932  2.329925
on campus        584  2.276164
with family      484  2.293285


The mean GPA actually dropped by a very small portion after cleaning, implying that GPA may have been the most accurately reported variable in the table and that the cleaning did not have a huge effect on it. One relationship that is interesting is definitely the housing cleaning, because rather than having so many different means, you can actually see that the off campus mean GPA with a larger sample size has a slightly higher GPA. We cannot draw any conclusions from it, but it is an interesting relationship. Cleaning changed the picture by showing us the true catagories of the different variables, and also helping gain a more accurate data point for overalls and averages

In [65]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
